# H3. Design, predictions and pre-registration

Fixed here, before any hardware run:
1. **the schedule**: which event runs in which job on which calibration day, for every arm;
2. **the analysis**: backtest settings, models, endpoints, decision rules;
3. **the predictions**: what each arm should show under a range of hardware fingerprint sizes, from an emulator.

It writes `outputs/prereg/` and prints a SHA-256 fingerprint of it and of the code. Email the fingerprint to
yourself (or post it) before session 1. That timestamps the plan.

### Arms (same 1,000 events in every arm; 8 calibration days x 2 jobs; 1,024 shots; every job carries all arms)

| Arm | Circuit | Assignment of events to jobs | Role |
|---|---|---|---|
| R | HSBC "longer", 129-qubit chain | random | honest test against the classical baseline |
| C | same | market day d on calibration day d, in market-time order | HSBC's natural pipeline |
| N | same gates, every data angle zero | as C | fingerprints; replay of any submission order |

Run time: about 3 hours the first time (emulator predictions, saved piece by piece, so an interrupted run resumes);
cached afterwards.

In [ ]:
N_SESSIONS, JOBS_PER_SESSION = 8, 2
SHOTS      = 1024
CAP_MIN    = 50.0
SIGMAS     = (0.0, 0.5, 1.0, 2.0)          # fingerprint size per feature, in shot-noise sd units
N_DRAWS    = 2
BLINDS     = (0, 1, 2)
EPS_CZ     = 0.0026                       # median CZ error on ibm_fez in the PQK100 pilot
RECOMPUTE  = False

def bt_settings(n):
    """Training window = 36% of the events (about 3 market days); test blocks = 4% (about a quarter day)."""
    return int(round(0.36 * n)), max(8, int(round(0.04 * n)))

In [ ]:
import os, sys, json, time, math, warnings
from pathlib import Path
# Find the study folder (the one holding the hsbcrep package), wherever Jupyter or VS Code started
def _find_root():
    starts = [Path(p).parent for p in [globals().get("__vsc_ipynb_file__")] if p]
    starts += [Path.cwd()]
    for s in starts:
        for p in [s, *s.parents]:
            if (p / "hsbcrep" / "__init__.py").exists():
                return p.resolve()
    raise RuntimeError(f"Cannot find the repository folder (the one holding hsbcrep/) from {Path.cwd()}. Set ROOT by hand in this cell.")
ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repository folder:", ROOT)
OUT = ROOT / "outputs"; OUT.mkdir(exist_ok=True)
RAW = ROOT / "data" / "raw"
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200)
import matplotlib.pyplot as plt
from hsbcrep import pqfm, backtest as BT, design as D
from hsbcrep.featuremap import cone_stats
BT.N_JOBS = 1

ev = pd.read_csv(OUT / "events.csv"); F = pd.read_csv(OUT / "features.csv")
hw = np.load(OUT / "hw_events.npy")
NL, NLsh, NULL = np.load(OUT / "noiseless_longer129_hw.npy"), np.load(OUT / "noiseless_shorter129_hw.npy"), np.load(OUT / "noiseless_null129.npy")
pos = {e: k for k, e in enumerate(hw)}
FAST = {k: BT.families()[k] for k in ("LR", "XGB")}

def score(X, e, blind, fams=FAST):
    o = np.argsort(ev.t.values[e], kind="stable")
    n_train, block = bt_settings(len(e))
    p = BT.rolling(X[o], ev.y.values[e][o], ev.t.values[e][o], ev.day.values[e][o], n_train=n_train, block=block, blind_days=blind, fams=fams)
    return BT.summarize(p)["pooled_auc"]

## H3.1 Schedule

In [ ]:
S = D.make_schedule(ev, hw, N_SESSIONS, JOBS_PER_SESSION, anchor=None, seed=11)   # no anchor: the null arm measures fingerprints
print(S.groupby("arm").size().to_dict(), "| rows per job:", S.groupby(["session", "job"]).size().min(), "to", S.groupby(["session", "job"]).size().max())
Rr = S[S.arm == "R"].assign(y=lambda d: ev.y.values[d.event])
print("R arm: fill rate per job ranges", Rr.groupby(["session", "job"]).y.mean().min().round(2), "to", Rr.groupby(["session", "job"]).y.mean().max().round(2))

## H3.2 Budget

Cost model from the PQK100 pilot on ibm_fez: 284 us per shot and 11.4 s per job, 3 measurement bases per row.
Session 0, a dry run of two short jobs (4 and 20 rows, about 45 seconds), re-measures both terms with HSBC's gate
twirling on.

**Pre-registered budget rule.** If, after session 0, the projected total exceeds 50 min, the null arm is halved
(every other slot of each job) for all sessions before session 1 runs. Nothing else changes.

In [ ]:
est = (len(S) * 3 * SHOTS * 284e-6 + N_SESSIONS * JOBS_PER_SESSION * 11.4) / 60
s0 = (24 * 3 * SHOTS * 284e-6 + 2 * 11.4) / 60
print(f"{len(S)} rows x 3 bases x {SHOTS} shots in {N_SESSIONS * JOBS_PER_SESSION} jobs: about {est:.1f} min, "
      f"plus {s0:.1f} min for session 0: {est + s0:.1f} min in total. Cap {CAP_MIN} min.")

## H3.3 Baselines on the 1,000 events (no hardware needed)

HSBC report AUC 0.63 for classical features and 0.60 for noiseless simulation. On random 1,000-event subsets of our
pool, the classical baseline ranges from 0.51 to 0.64 (sd 0.05), so all comparisons are paired on the same events.

In [ ]:
f_base = OUT / "prereg_baselines.csv"
if f_base.exists() and not RECOMPUTE:
    base = pd.read_csv(f_base)
else:
    rows = []
    for name, X in (("classical 216", F.values[hw]), ("noiseless longer", NL.reshape(len(hw), -1)),
                    ("noiseless shorter", NLsh.reshape(len(hw), -1)), ("time only", ((ev.t.values[hw] - ev.t.values.min()) / 8.64e7)[:, None])):
        for blind in BLINDS:
            rows.append({"input": name, "blind_days": blind, "auc": score(X, hw, blind)})
            print(rows[-1], flush=True)
    base = pd.DataFrame(rows); base.to_csv(f_base, index=False)
base.pivot(index="input", columns="blind_days", values="auc").round(3)

## H3.4 Emulator

Hardware features = retention x noiseless + per-job and per-day fingerprints + gate noise + shot noise.
Retention per qubit = (1 - eps)^(CZ gates in its light cone). The fingerprint size sigma is unknown; predictions are
given for sigma = 0 to 2 shot-noise units. Each row's fingerprint comes from its own (session, job).

In [ ]:
ret = (1 - EPS_CZ) ** cone_stats(129, pqfm.pqfm(129, 216, "longer"))[2]
print(f"predicted signal retention on ibm_fez, median over qubits: {np.median(ret):.2f}")
Sc = S[~S.anchor].reset_index(drop=True)
E = np.stack([NULL if nl else NL[pos[e]] for e, nl in zip(Sc.event, Sc.null)])

## H3.5 Predictions

For each sigma: R and C at blinding 0; N (in C's placement) at blinding 0 and 1; and the replay of N's slots under an
outcome-sorted submission order (the class-ordered file of do Carmo et al.).

In [ ]:
f_pred = OUT / "prereg_predictions.csv"
parts = OUT / "prereg_pred_parts"                       # one file per (sigma, draw): an interrupted run resumes here
if f_pred.exists() and not RECOMPUTE:
    pred = pd.read_csv(f_pred)
else:
    parts.mkdir(exist_ok=True)
    tasks = [(sig, draw) for sig in SIGMAS for draw in range(N_DRAWS)]
    w = os.environ.get("H3_WORKER")                     # optional "k/m": run every m-th task from k (parallel workers)
    mine = tasks if not w else tasks[int(w.split("/")[0])::int(w.split("/")[1])]
    for sig, draw in mine:
        f_part = parts / f"sigma{sig}_draw{draw}.csv"
        if f_part.exists() and not RECOMPUTE:
            continue
        rows = []
        H = D.emulate(E, Sc, ret, SHOTS, sigma_job=sig, sigma_day=sig, seed=100 * draw + int(10 * sig))
        for arm, blinds in (("R", (0,)), ("C", (0,)), ("N", (0, 1))):
            m = (Sc.arm == arm).values
            for blind in blinds:
                rows.append({"sigma": sig, "draw": draw, "arm": arm, "blind_days": blind,
                             "auc": score(H[m].reshape(m.sum(), -1), Sc.event.values[m], blind)})
        Nr = Sc[Sc.arm == "N"].reset_index().rename(columns={"index": "row"})
        pairs = D.reassign(Nr, ev, hw, "by outcome", np.random.default_rng(draw))
        rows.append({"sigma": sig, "draw": draw, "arm": "N replayed by outcome", "blind_days": 0,
                     "auc": score(H[[r for r, _ in pairs]].reshape(len(pairs), -1), np.array([e for _, e in pairs]), 0)})
        pd.DataFrame(rows).to_csv(f_part, index=False)
        print(f"sigma {sig}, draw {draw} done", flush=True)
    done = [parts / f"sigma{sig}_draw{draw}.csv" for sig, draw in tasks]
    if not all(f.exists() for f in done):
        raise SystemExit(f"{sum(f.exists() for f in done)} of {len(done)} prediction parts done; run this cell again")
    pred = pd.concat([pd.read_csv(f) for f in done], ignore_index=True); pred.to_csv(f_pred, index=False)
pred.groupby(["arm", "blind_days", "sigma"]).auc.mean().unstack("sigma").round(3)

Table for the paper (`paper/tables/predictions.tex`), written from the numbers above.

In [ ]:
tab = pred.groupby(["arm", "blind_days", "sigma"]).auc.mean().unstack("sigma")
b0 = base[base.blind_days == 0].set_index("input").auc
names = {("R", 0): "R (random)", ("C", 0): "C (natural pipeline)", ("N", 0): "N (no data), 0-day blinding",
         ("N", 1): "N (no data), 1-day blinding", ("N replayed by outcome", 0): "N replayed, sorted by outcome"}
lines = [r"\begin{table}[h]", r"\centering\small", r"\begin{tabular}{l" + "r" * len(SIGMAS) + "}", r"\toprule",
         "Arm & " + " & ".join(f"$\\sigma={s:g}$" for s in SIGMAS) + r" \\", r"\midrule"]
for key, label in names.items():
    if key in tab.index:
        lines.append(label + " & " + " & ".join(f"{tab.loc[key, s]:.3f}" for s in SIGMAS) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}",
          (r"\caption{Emulator predictions of pooled out-of-sample AUC (HSBC rolling backtest, 1{,}000 events). $\sigma$ is the "
           r"per-feature fingerprint size per job and per day, in units of the shot-noise standard deviation. Baselines on "
           f"the same events: classical {b0.get('classical 216', float('nan')):.3f}, noiseless ``longer'' "
           f"{b0.get('noiseless longer', float('nan')):.3f}, noiseless ``shorter'' {b0.get('noiseless shorter', float('nan')):.3f}, "
           f"time only {b0.get('time only', float('nan')):.3f}." + "}"),
          r"\label{tab:pred}", r"\end{table}"]
paper_tab = ROOT / "paper" / "tables"; paper_tab.mkdir(parents=True, exist_ok=True)
(paper_tab / "predictions.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## H3.6 Endpoints and decision rules (frozen below)

AUC is the pooled out-of-sample ROC AUC of HSBC's rolling backtest: training window 360 events (about 3 market days),
test blocks of 40, logistic regression and gradient boosting, the family with the best in-sample CV AUC predicts each
block. Paired differences are bootstrapped over test events (2,000 resamples).

**Primary endpoints (blinding window 0 days; the same 1,000 events in every comparison)**
1. **Hardware gain under random assignment**: AUC(R) - AUC(classical). Declared if the 95% CI lies above +0.05.
   HSBC report +0.34.
2. **Natural-pipeline leak**: AUC(C) - AUC(R). Declared if the 95% CI lies above +0.05.
3. **Leak without data**: AUC(N) - 0.5. Declared if the 95% CI lies above +0.05.

**Secondary**: hardware vs noiseless; blinding windows 0, 1, 2 days; fingerprint sizes within a job, between jobs on
a day and between days, from N; replay of N's slots under random, chronological, by-symbol and
by-outcome orders; all four HSBC model families with their grids; measured vs predicted at the measured sigma.

**Reading of outcomes**
* 1 no, 2 no, 3 no: on Heron r2, with controlled assignment, hardware features give no gain; the measured sigma and
  the replay show what other submission orders would have produced.
* 1 yes: a gain that survives random assignment; it is then tested against the static and drift emulators.
* 2 or 3 yes: the natural pipeline alone leaks on this device; its size is compared with the replay.

## H3.7 Freeze

The first run writes `outputs/prereg/` and its fingerprint. Later runs only check it, so the fingerprint you sent
stays valid. To re-freeze on purpose (before any hardware session), delete `outputs/prereg/` and run again.

In [ ]:
pre = OUT / "prereg"
if not (pre / "SHA256.txt").exists():
    pre.mkdir(exist_ok=True)
    S.to_csv(pre / "schedule.csv", index=False)
    pred.to_csv(pre / "predictions.csv", index=False); base.to_csv(pre / "baselines.csv", index=False)
    plan = {"created_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), "shots": SHOTS, "sessions": N_SESSIONS,
            "jobs_per_session": JOBS_PER_SESSION, "cap_min": CAP_MIN, "budget_rule": "halve the null arm before session 1 if projected > cap",
            "backtest": {"train_fraction": 0.36, "block_fraction": 0.04, "blinds": list(BLINDS),
                         "families_primary": ["LR", "XGB"], "families_secondary": ["LR", "XGB", "RF", "NN"]},
            "mitigation": "Pauli twirling on gates and readout + TREX, no ZNE (as HSBC)", "device": "ibm_fez",
            "chain_qubits": 129, "anchor": None, "label_rule": json.loads((OUT / "label_rule.json").read_text()),
            "decision_rules": "see H3.6"}
    (pre / "plan.json").write_text(json.dumps(plan, indent=1))
    (pre / "SHA256.txt").write_text(D.prereg_digest(ROOT) + "\n")
    print("frozen now.")
frozen = (pre / "SHA256.txt").read_text().strip()
now = D.prereg_digest(ROOT)
print("pre-registration fingerprint (SHA-256):", frozen)
print("check:", "matches" if now == frozen else "DOES NOT MATCH: the frozen files or the code changed")